# Smart IoT Energy Meter — AI/ML Pipeline
**Based on:** ESP32 + ACS712 (current) + ZMPT101B (voltage) + Blynk 2.0

## Pipeline Overview
1. Synthetic data generation (mimics real ESP32 sensor readings)
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model 1: **Energy Consumption Forecasting** (XGBoost Regressor)
5. Model 2: **Anomaly Detection** (Isolation Forest)
6. Model export for API serving

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='darkgrid')
print('Libraries loaded successfully!')

## 1. Synthetic Data Generation
Simulates 6 months of 15-minute interval ESP32 readings.

In [ ]:
def generate_energy_data(n_days=180, freq_minutes=15):
    """Generate realistic ESP32 IoT energy meter readings."""
    periods = int(n_days * 24 * 60 / freq_minutes)
    timestamps = pd.date_range(start='2025-01-01', periods=periods, freq=f'{freq_minutes}min')

    # Time-based features
    hour = timestamps.hour
    dow  = timestamps.dayofweek  # 0=Monday
    month = timestamps.month

    # Voltage: ~220V AC with small fluctuations (ZMPT101B sensor range)
    voltage = 220 + np.random.normal(0, 3, periods) + 2 * np.sin(2 * np.pi * hour / 24)

    # Current: usage pattern — morning/evening peaks (ACS712 sensor)
    base_current = (
        0.5
        + 1.5 * ((hour >= 7)  & (hour <= 9)).astype(float)   # morning peak
        + 2.0 * ((hour >= 18) & (hour <= 22)).astype(float)  # evening peak
        + 0.3 * ((dow >= 5)).astype(float)                   # weekend boost
        + 0.2 * ((month >= 5) & (month <= 8)).astype(float)  # summer AC load
    )
    current = base_current + np.abs(np.random.normal(0, 0.3, periods))

    # Power (W) = V * I * power_factor
    power_factor = np.random.uniform(0.85, 0.99, periods)
    power = voltage * current * power_factor

    # Energy (kWh) per interval
    energy_kwh = power * (freq_minutes / 60) / 1000

    # Cost (INR) at ₹7/kWh (typical Indian tariff)
    cost = energy_kwh * 7.0

    # Temperature (°C) — affects load
    temperature = 25 + 8 * np.sin(2 * np.pi * (month - 3) / 12) + np.random.normal(0, 1.5, periods)

    # Inject ~2% anomalies (voltage spikes, current surges, power outages)
    anomaly_idx = np.random.choice(periods, size=int(0.02 * periods), replace=False)
    anomaly_type = np.random.choice(['voltage_spike','current_surge','dropout'], size=len(anomaly_idx))
    is_anomaly = np.zeros(periods, dtype=int)
    for idx, atype in zip(anomaly_idx, anomaly_type):
        is_anomaly[idx] = 1
        if atype == 'voltage_spike':
            voltage[idx] *= np.random.uniform(1.15, 1.30)
        elif atype == 'current_surge':
            current[idx] *= np.random.uniform(2.0, 4.0)
        elif atype == 'dropout':
            voltage[idx] *= 0.0
            current[idx] *= 0.0

    df = pd.DataFrame({
        'timestamp': timestamps,
        'voltage_v': np.round(voltage, 2),
        'current_a': np.round(current, 3),
        'power_w': np.round(power, 2),
        'energy_kwh': np.round(energy_kwh, 4),
        'power_factor': np.round(power_factor, 3),
        'temperature_c': np.round(temperature, 1),
        'cost_inr': np.round(cost, 4),
        'is_anomaly': is_anomaly
    })
    return df

df = generate_energy_data()
df.to_csv('energy_data.csv', index=False)
print(f'Dataset shape: {df.shape}')
df.head(10)

## 2. Exploratory Data Analysis

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
cols = ['voltage_v', 'current_a', 'power_w', 'energy_kwh', 'power_factor', 'temperature_c']
for ax, col in zip(axes.flat, cols):
    df[col].hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=120)
plt.show()

In [ ]:
# Daily energy consumption
daily = df.groupby(df['timestamp'].dt.date)['energy_kwh'].sum().reset_index()
daily.columns = ['date', 'daily_kwh']

fig = px.line(daily, x='date', y='daily_kwh',
              title='Daily Energy Consumption (kWh)',
              labels={'daily_kwh': 'Energy (kWh)', 'date': 'Date'})
fig.update_traces(line_color='#FF6B35')
fig.show()

In [ ]:
# Hourly average power
df['hour'] = df['timestamp'].dt.hour
hourly_power = df.groupby('hour')['power_w'].mean().reset_index()

fig = px.bar(hourly_power, x='hour', y='power_w',
             title='Average Power by Hour of Day',
             labels={'power_w': 'Avg Power (W)', 'hour': 'Hour'},
             color='power_w', color_continuous_scale='Viridis')
fig.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(9, 7))
corr = df[['voltage_v','current_a','power_w','energy_kwh','power_factor','temperature_c','cost_inr']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=120)
plt.show()

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour']          = df['timestamp'].dt.hour
    df['day_of_week']   = df['timestamp'].dt.dayofweek
    df['month']         = df['timestamp'].dt.month
    df['is_weekend']    = (df['day_of_week'] >= 5).astype(int)
    df['is_peak_hour']  = df['hour'].isin([7,8,9,18,19,20,21,22]).astype(int)
    df['hour_sin']      = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']      = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin']     = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']     = np.cos(2 * np.pi * df['month'] / 12)
    df['apparent_power']= df['voltage_v'] * df['current_a']
    df['reactive_power']= df['apparent_power'] * np.sqrt(1 - df['power_factor']**2)

    # Rolling stats (lag features)
    df['power_roll_mean_4']  = df['power_w'].rolling(4,  min_periods=1).mean()
    df['power_roll_mean_24'] = df['power_w'].rolling(24, min_periods=1).mean()
    df['power_roll_std_4']   = df['power_w'].rolling(4,  min_periods=1).std().fillna(0)
    df['energy_lag_1']       = df['energy_kwh'].shift(1).fillna(0)
    df['energy_lag_4']       = df['energy_kwh'].shift(4).fillna(0)
    df['energy_lag_96']      = df['energy_kwh'].shift(96).fillna(0)  # same time yesterday
    return df

df_feat = engineer_features(df)
print(f'Features after engineering: {df_feat.shape[1]}')
df_feat.head(3)

## 4. Model 1 — Energy Consumption Forecasting (XGBoost)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

FORECAST_FEATURES = [
    'voltage_v', 'current_a', 'power_factor', 'temperature_c',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'is_weekend', 'is_peak_hour', 'apparent_power', 'reactive_power',
    'power_roll_mean_4', 'power_roll_mean_24', 'power_roll_std_4',
    'energy_lag_1', 'energy_lag_4', 'energy_lag_96'
]
TARGET = 'energy_kwh'

X = df_feat[FORECAST_FEATURES].values
y = df_feat[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_sc, y_train, eval_set=[(X_test_sc, y_test)], verbose=50)

y_pred = xgb_model.predict(X_test_sc)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2   = r2_score(y_test, y_pred)
print(f'MAE:  {mae:.5f} kWh')
print(f'RMSE: {rmse:.5f} kWh')
print(f'R²:   {r2:.4f}')

# Save model + scaler
joblib.dump(xgb_model, 'energy_forecast_model.pkl')
joblib.dump(scaler,    'energy_forecast_scaler.pkl')
joblib.dump(FORECAST_FEATURES, 'forecast_features.pkl')
print('Saved: energy_forecast_model.pkl, energy_forecast_scaler.pkl')

In [ ]:
# Feature importance
fi = pd.Series(xgb_model.feature_importances_, index=FORECAST_FEATURES).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
fi.plot(kind='barh', color='teal')
plt.title('XGBoost Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

In [ ]:
# Actual vs Predicted
n = 200
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test[:n], name='Actual',    line=dict(color='blue')))
fig.add_trace(go.Scatter(y=y_pred[:n], name='Predicted', line=dict(color='red', dash='dash')))
fig.update_layout(title='Energy Forecast — Actual vs Predicted (first 200 points)',
                  xaxis_title='Sample', yaxis_title='Energy (kWh)')
fig.show()

## 5. Model 2 — Anomaly Detection (Isolation Forest)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

ANOMALY_FEATURES = ['voltage_v', 'current_a', 'power_w', 'power_factor',
                    'apparent_power', 'reactive_power', 'power_roll_std_4']

X_anom = df_feat[ANOMALY_FEATURES].values
anom_scaler = StandardScaler()
X_anom_sc   = anom_scaler.fit_transform(X_anom)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_anom_sc)

# -1 = anomaly, 1 = normal → convert to 1/0
raw_pred  = iso_forest.predict(X_anom_sc)
anom_pred = (raw_pred == -1).astype(int)
scores    = -iso_forest.score_samples(X_anom_sc)  # higher = more anomalous

df_feat['anomaly_pred']  = anom_pred
df_feat['anomaly_score'] = np.round(scores, 4)

print('Classification Report:')
print(classification_report(df_feat['is_anomaly'], anom_pred, target_names=['Normal','Anomaly']))

# Save anomaly model
joblib.dump(iso_forest,     'anomaly_model.pkl')
joblib.dump(anom_scaler,    'anomaly_scaler.pkl')
joblib.dump(ANOMALY_FEATURES, 'anomaly_features.pkl')
print('Saved: anomaly_model.pkl, anomaly_scaler.pkl')

In [ ]:
# Confusion matrix
cm = confusion_matrix(df_feat['is_anomaly'], anom_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Normal','Pred Anomaly'],
            yticklabels=['True Normal','True Anomaly'])
plt.title('Anomaly Detection — Confusion Matrix')
plt.tight_layout()
plt.savefig('anomaly_confusion.png', dpi=120)
plt.show()

In [ ]:
# Visualize anomalies on power timeline
sample = df_feat.iloc[:2000].copy()
fig = px.scatter(sample, x='timestamp', y='power_w',
                 color=sample['anomaly_pred'].map({0:'Normal',1:'Anomaly'}),
                 color_discrete_map={'Normal':'steelblue','Anomaly':'red'},
                 title='Power Readings with Detected Anomalies',
                 labels={'power_w': 'Power (W)'})
fig.show()

## 6. Monthly Cost Summary

In [ ]:
df_feat['year_month'] = df_feat['timestamp'].dt.to_period('M').astype(str)
monthly = df_feat.groupby('year_month').agg(
    total_kwh   = ('energy_kwh', 'sum'),
    total_cost  = ('cost_inr',   'sum'),
    avg_voltage = ('voltage_v',  'mean'),
    avg_current = ('current_a',  'mean'),
    anomalies   = ('anomaly_pred','sum')
).round(2).reset_index()

fig = px.bar(monthly, x='year_month', y='total_kwh',
             text='total_kwh', title='Monthly Energy Consumption (kWh)',
             color='total_cost', color_continuous_scale='RdYlGn_r')
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.show()
print(monthly)

In [ ]:
# Save final processed dataset
df_feat.to_csv('energy_data_processed.csv', index=False)
print('All models and data saved. Ready for API + Streamlit deployment!')